# Tests 2 & 4: MD Stability and Condensed-Phase Water

Colab-runnable driver for Test 2 (short-MD molecular stability, the
paper's real 349-atom drug-like benchmark molecule) and Test 4
(condensed-phase 168-water box). Separate from `01_dimer_scan.ipynb`
(Test 3) -- different environment stack (`md`, not `ani-mace`/`uma`) and
different scope.

**Read `RESULTS.md`'s "Test 2 and Test 4" section before running this** --
it documents every way this session's protocol deviates from Ranasinghe
et al. 2025's actual Test 2/4 setup (shortened production lengths, no
solute in Test 4, `inference_settings="turbo"` never having been
exercised anywhere but here, etc.). Those deviations are compute-budget
and scope decisions already made -- this notebook just runs them.

**Both tests are resumable.** If Colab disconnects, just re-run the same
cell -- `run_resumable_md` picks up from the last checkpoint.

**Use Google Drive for `MD_ROOT`** so results/checkpoints survive a
runtime recycle -- see the Drive-mount cell below. This is NOT optional
for Test 4 in particular: 175 ps of real MD on ~500 atoms will not finish
in one Colab session, and losing checkpoints on disconnect would waste
the compute-unit budget this work is explicitly scoped against.

## 1. Mount Google Drive, authenticate to GitHub, and clone the repo

This repo (`SMLion1959422/mlip-audit`) is **private**, so a fresh Colab
runtime needs its own credential to clone it -- separate from any
authentication on your local machine. Recommended: a short-lived,
**read-only**, repo-scoped GitHub Personal Access Token (PAT). Colab only
ever needs to READ this repo here, never push to it -- all Test 2/4
output goes to Google Drive, not back to GitHub (see "Record results" at
the end).

To create one:
1. Go to https://github.com/settings/personal-access-tokens/new
2. **Repository access** -> "Only select repositories" -> `mlip-audit`.
3. **Permissions** -> **Repository permissions** -> **Contents: Read-only**
   (leave everything else at "No access").
4. **Expiration**: 7 days is plenty for one Colab session.
5. Generate, then copy the token (starts with `github_pat_...`).

The clone cell below prompts for it with a hidden input (`getpass`) -- it
is never printed, never written to Drive/disk, and is deleted from memory
immediately after cloning.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# All Test 2/4 checkpoints and results go here -- survives a runtime recycle.
os.environ['MLIP_AUDIT_MD_ROOT'] = '/content/drive/MyDrive/mlip-audit-md'
os.makedirs(os.environ['MLIP_AUDIT_MD_ROOT'], exist_ok=True)
print('MD_ROOT set to:', os.environ['MLIP_AUDIT_MD_ROOT'])

In [ ]:
import subprocess
import getpass

REPO_OWNER = "SMLion1959422"
REPO_NAME = "mlip-audit"
REPO_DIR = "/content/mlip-audit"

if not os.path.isdir(REPO_DIR):
    token = getpass.getpass("Paste your read-only GitHub PAT (hidden input): ")
    clone_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    try:
        subprocess.run(
            ["git", "clone", clone_url, REPO_DIR],
            check=True, capture_output=True, text=True,
        )
        print("Cloned OK.")
    except subprocess.CalledProcessError as e:
        # Redact the token before printing -- it's embedded in the URL,
        # and CalledProcessError's stderr may otherwise echo it back.
        print("git clone failed (stderr, token redacted):")
        print(e.stderr.replace(token, "<REDACTED>"))
        raise RuntimeError("git clone failed -- see redacted stderr above") from None
    finally:
        del token, clone_url  # don't keep the credential in memory any longer than needed
else:
    print(f"{REPO_DIR} already exists -- skipping clone "
          "(safe to re-run this whole notebook after a disconnect).")

%cd $REPO_DIR

In [ ]:
!bash setup.sh md

## 2. HuggingFace login (only if setup.sh reported no cached token)

Required for UMA-S and both eSEN checkpoints. Make sure you've accepted
both gated model licenses on huggingface.co with this account.

In [ ]:
from huggingface_hub import get_token

if get_token() is None:
    from huggingface_hub import login
    login()  # interactive widget; do not pass token= here
else:
    print("Already logged in.")

## 3. Verify `inference_settings="turbo"` actually works here

This has NEVER been exercised successfully anywhere in this project --
only tested locally on a machine without a C++ compiler, where it fails.
Colab normally has gcc, so it should work, but VERIFY before trusting any
downstream result. If this cell errors, fall back to
`inference_settings="batch"` in `mlip_audit/models.py` (slower, but the
one mode validated so far in this project) and re-run.

In [ ]:
from mlip_audit.models import get_calc, prepare_atoms_for_model
from mlip_audit.molecules import build_molecule_from_smiles

atoms = build_molecule_from_smiles("CCO", seed=0)  # ethanol, cheap
calc = get_calc("uma-s-1p1", inference_settings="turbo")
atoms.calc = calc
prepare_atoms_for_model(atoms, "uma-s-1p1", charge=0, spin=1)
e = atoms.get_potential_energy()
print("turbo mode energy:", e, "eV -- if this printed with no error, turbo works here.")

## 4. Test 2: short-MD molecular stability

**4a** below is a cheap dress rehearsal of the interrupt/resume mechanism
on UMA-S only, before trusting it with the real 100 ps x 4-model budget
in **4b**. Run 4a first.

### 4a. Dress rehearsal -- verify the interrupt/resume seam (UMA-S only)

This calls `test2_md_stability.run_one()` directly (not the CLI), with
the total time and checkpoint interval temporarily shrunk to 2 ps / 0.2 ps
just for this rehearsal, and pointed at a **separate** `MD_ROOT`
(`...-rehearsal`) so it can never collide with or corrupt the real Test 2
checkpoints in 4b. Neither override touches `mlip_audit/config.py`.

**How to run it:**
1. Run the cell below. There will be a quiet pause with no output during
   LBFGS pre-optimization (its own per-step logging is intentionally off,
   same as production) -- this is expected, not a hang.
2. Wait for the log line `Starting md.traj fresh: 2.000 ps target ...`.
   **Only interrupt after this line appears** -- interrupting during the
   quiet LBFGS pause just restarts LBFGS on re-run (fresh geometry, no
   `md.traj` written yet), which doesn't exercise the resume path at all.
3. Once you see that line, click the cell's Stop (square) button to
   simulate a Colab disconnect mid-MD.
4. Re-run the *same* cell, unchanged. It should now log
   `Resuming md.traj from frame N (X.XXX ps elapsed) of 2.000 ps target`
   instead of starting fresh, and run to completion (or interrupt it again
   partway if you want to test a second seam).
5. Run the verification cell after that.

In [ ]:
import logging
from pathlib import Path

import mlip_audit.test2_md_stability as t2
from mlip_audit.config import MD_ROOT as _PRODUCTION_MD_ROOT

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,  # override any logging config Colab's own imports may have set
)

# Separate MD_ROOT -- this rehearsal never touches the real Test 2 checkpoints.
# Derived from the PRODUCTION root fresh each time (not from t2.MD_ROOT,
# which this cell itself mutates) so re-running this cell to resume lands
# on the exact same rehearsal directory instead of drifting to a new one.
t2.MD_ROOT = Path(str(_PRODUCTION_MD_ROOT) + "-rehearsal")

# REHEARSAL-ONLY overrides (small + fast). These reassign globals inside
# the already-imported test2_md_stability module, not mlip_audit.config --
# the real 100 ps sweep in 4b re-reads the real values fresh and is unaffected.
t2.TEST2_TOTAL_TIME_PS = 2.0
t2.TEST2_CHECKPOINT_EVERY_PS = 0.2
t2.TEST2_TRAJ_SAVE_EVERY_PS = 0.2

print("Rehearsal MD_ROOT:", t2.MD_ROOT)
print("Waiting on LBFGS pre-optimization (no output expected here)...\n")

t2.run_one("drug_like_benchmark", "uma-s-1p1", seed=0)

In [ ]:
# Verify the interrupt/resume seam: read back the rehearsal trajectory and
# log, and check for the same kinds of corruption this project has audited
# for before (duplicate/truncated/out-of-order entries) -- applied here to
# the interruption boundary specifically. Run this AFTER the rehearsal
# cell above has reached its 2 ps target (after at least one interrupt +
# resume cycle).
import pandas as pd
from ase.io.trajectory import Trajectory

rehearsal_dir = t2.MD_ROOT / "test2" / "drug_like_benchmark" / "uma-s-1p1" / "seed0"
traj_path = rehearsal_dir / "md.traj"
log_path = rehearsal_dir / "md.log"

log_df = pd.read_csv(log_path)
print(f"log rows: {len(log_df)}")
print(log_df.to_string())

# time_ps must be strictly increasing with constant spacing
# (TEST2_TRAJ_SAVE_EVERY_PS) and no duplicates -- a resume that mishandled
# the seam would show either a repeated timestamp (double-wrote the
# pre-interrupt frame) or a gap (lost a frame / mis-set the resume offset).
expected_dt = t2.TEST2_TRAJ_SAVE_EVERY_PS
dt = log_df["time_ps"].diff().dropna()
bad_spacing = dt[(dt - expected_dt).abs() > 1e-6]
n_duplicates = log_df["time_ps"].duplicated().sum()

print(f"\nExpected spacing: {expected_dt} ps")
print("Rows with wrong spacing:",
      "OK (none)" if bad_spacing.empty else f"BAD -- {bad_spacing.tolist()}")
print("Duplicate timestamps:",
      "OK (none)" if n_duplicates == 0 else f"BAD -- {n_duplicates}")

with Trajectory(str(traj_path), "r") as traj:
    n_frames = len(traj)
print(f"\n.traj frame count: {n_frames} (log row count: {len(log_df)})",
      "-- OK" if n_frames == len(log_df) else "-- BAD, MISMATCH")

# Energy continuity across the interruption boundary -- the resumed run
# continues from the exact checkpointed positions/momenta, so E_tot should
# show only normal thermostat noise, never a discontinuous jump right at
# the seam.
e_tot = log_df["E_pot_eV"] + log_df["E_kin_eV"]
print("\nE_tot_eV by frame (scan for a large jump at any one row):")
print(e_tot.to_string())

all_ok = bad_spacing.empty and n_duplicates == 0 and n_frames == len(log_df)
print("\nALL CHECKS PASSED -- seam looks clean." if all_ok else
      "\nSOMETHING FAILED -- do not trust the real sweep's resumability yet.")

### 4b. Full sweep (UMA-S + eSEN-conserving + eSEN-direct + ANI-2x)

Only run this once 4a's verification cell prints "ALL CHECKS PASSED".

In [ ]:
# Full sweep: 1 molecule (the paper's real 349-atom drug-like benchmark)
# x 4 models x 1 seed = 4 runs, each 100 ps. See RESULTS.md's Test 2
# deviations table and "Test 2 molecule: SMILES provenance and
# verification" section for how this molecule was obtained/validated.
# Resumable -- safe to re-run this exact cell after any disconnect.
# To run a subset (e.g. while iterating), pass --molecule/--model/--seed.
!python -m mlip_audit.test2_md_stability --verbose

In [ ]:
# Bond-length stability analysis across the full sweep.
from mlip_audit.config import MD_ROOT, TEST2_MOLECULES, TEST2_MODELS, TEST2_SEEDS
from mlip_audit.md_analysis import bond_length_trajectory_report

for molecule in TEST2_MOLECULES:
    for model in TEST2_MODELS:
        for seed in TEST2_SEEDS:
            traj_path = MD_ROOT / "test2" / molecule / model / f"seed{seed}" / "md.traj"
            if not traj_path.exists():
                print(f"{molecule}/{model}/seed{seed}: NOT RUN YET")
                continue
            report = bond_length_trajectory_report(traj_path)
            flag = "UNSTABLE" if report["any_unstable"] else "stable"
            print(f"{molecule}/{model}/seed{seed}: {report['n_frames']} frames, {flag}")
            if report["any_unstable"]:
                for b in report["bonds"]:
                    if b["flagged_unstable"]:
                        print(f"    bond {b['symbols']} (atoms {b['i']},{b['j']}): "
                              f"d0={b['d0']:.3f} A, max={b['d_max']:.3f} A, ratio={b['max_ratio']:.2f}x")

## 5. Test 4: condensed-phase water

In [ ]:
# Full sweep: 4 models, each 125 ps NVT + 50 ps NPT. Resumable per-phase.
# This is the most compute-intensive part of this notebook -- consider
# running one model at a time (--model uma-s-1p1, etc.) across multiple
# sessions if a single session's time/compute-unit budget is tight.
!python -m mlip_audit.test4_condensed_water --verbose

In [ ]:
# O-O radial distribution function from each model's NPT production phase.
import matplotlib.pyplot as plt
from mlip_audit.config import MD_ROOT, TEST4_MODELS
from mlip_audit.md_analysis import oo_radial_distribution_function

fig, ax = plt.subplots(figsize=(8, 6))
for model in TEST4_MODELS:
    npt_traj = MD_ROOT / "test4" / model / "npt" / "npt.traj"
    if not npt_traj.exists():
        print(f"{model}: NPT not run yet")
        continue
    rdf = oo_radial_distribution_function(npt_traj)
    ax.plot(rdf["r"], rdf["g_r"], label=f"{model} ({rdf['n_frames_used']} frames)")

ax.axvline(2.8, color="gray", linestyle="--", linewidth=1, label="experimental O-O peak (~2.8 A)")
ax.set_xlabel("r (A)")
ax.set_ylabel("g_OO(r)")
ax.set_title("Test 4: O-O radial distribution function")
ax.legend()
ax.grid(True, alpha=0.3)
fig.savefig(str(MD_ROOT / "test4" / "rdf_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Record results

Results already live on Google Drive (`MD_ROOT`), so there's no separate
download step needed here (unlike Test 3's notebook) -- but do copy the
key findings (bond-stability table, RDF plot, any instabilities) into
`RESULTS.md`'s "Test 2 and Test 4" section by hand once a run completes,
the same way every other result in this project has been documented.